# Administrative boundaries — Natural Earth & US Census TIGER

Beyond per-country geoBoundaries, the admin backend serves **global cultural layers**
([Natural Earth](https://www.naturalearthdata.com/)) and **US Census TIGER/Line**
cartographic boundaries. This notebook maps both and shows the automatic CRS
normalisation to EPSG:4326.

## Setup

In [ ]:
import matplotlib.pyplot as plt
from earthlens.core import EarthLens

## Natural Earth — world countries

Natural Earth ships cultural layers at three scales (`10m` / `50m` / `110m`). The
`scale=` selector is optional and defaults per dataset (`110m` for countries). We
use the coarse `110m` layer here — it is tiny (~200 KB) and fast.

In [ ]:
countries = EarthLens(
    data_source='natural-earth',
    variables=['natural_earth:countries'],
    scale='110m',
).download(progress_bar=False)
print(len(countries), 'countries, EPSG:', countries.crs.to_epsg())

In [ ]:
ax = countries.plot(edgecolor='white', facecolor='#9ecae1', figsize=(11, 5.5))
ax.set_title('Natural Earth — admin-0 countries (110m)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()

## US Census TIGER — states

TIGER cartographic-boundary files arrive in **NAD83 (EPSG:4269)**; the backend
reprojects them to EPSG:4326 so every admin result shares one CRS. The `year=`
selector is optional (defaults to the dataset's vintage).

In [ ]:
states = EarthLens(
    data_source='tiger',
    variables=['tiger:state'],
).download(progress_bar=False)
print(len(states), 'state features, EPSG:', states.crs.to_epsg())

The 56 features include the 50 states plus DC and territories. We clip the plot to
the contiguous US for readability.

In [ ]:
ax = states.plot(edgecolor='black', facecolor='#fdd0a2', figsize=(10, 6))
ax.set_xlim(-128, -65)
ax.set_ylim(23, 50)
ax.set_title('US Census TIGER — states (reprojected NAD83 -> EPSG:4326)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()

## A note on CGAZ

The fourth source, **CGAZ** (Comprehensive Global Admin Zones), provides *seamless
global* ADM0/1/2 layers — every country in one file. Those files are large
(160–550 MB), so they are not fetched in this tutorial; use
`variables=['cgaz:adm0']` (no selector) when you need the seamless global layer.
It returns polygons in EPSG:4326 like the others.

## Takeaway

- Natural Earth (`scale=`) and TIGER (`year=` / `state=`) follow the same
  `download()` shape as geoBoundaries.
- The backend normalises every source to **EPSG:4326** — TIGER's NAD83 is
  reprojected automatically.
- `aggregate=` is rejected for all admin datasets: boundaries are vector polygons,
  not a gridded field.